<a href="https://colab.research.google.com/github/gracey25/us-foreign-aid-democracy-analysis/blob/main/notebooks/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd
import numpy as np

# ==========================================
# LOAD RAW DATASETS
# ==========================================
# GitHub raw URLs
VDEM_URL = "https://raw.githubusercontent.com/gracey25/us-foreign-aid-democracy-analysis/main/data/raw/vdem_1999_2023_subset.csv"
AID_URL = "https://raw.githubusercontent.com/gracey25/us-foreign-aid-democracy-analysis/main/data/raw/us_foreign_aid_usg_sector.csv"

# Read datasets into pandas DataFrames
df_vdem_raw = pd.read_csv(VDEM_URL)
df_aid_raw = pd.read_csv(AID_URL)

# ==========================================
# VERIFY ROW COUNTS & MEMORY USAGE
# ==========================================
def inspect_dataset(df, name):
    rows, cols = df.shape
    # Calculate memory usage in Megabytes (MB)
    memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

    print(f"--- {name} Dataset Inspection ---")
    print(f"Row Count:     {rows:,}")
    print(f"Column Count:  {cols}")
    print(f"Memory Usage:  {memory_mb:.2f} MB")
    print("\nColumn Data Types & Non-Null Counts:")
    print(df.info())
    print("\nHead Preview:")
    display(df.head(3))
    print("=" * 50 + "\n")

# Run inspection for both datasets
inspect_dataset(df_vdem_raw, "V-Dem (Democracy)")
inspect_dataset(df_aid_raw, "ForeignAssistance.gov (Aid Obligations)")


# ==========================================
# CLEAN V-DEM DATASET
# ==========================================

# Total missing values per column
missing_summary = pd.DataFrame({
    'Missing Count': df_vdem_raw.isnull().sum(),
    'Missing %': (df_vdem_raw.isnull().mean() * 100).round(2)
})
print("--- Missing Values Summary ---")
print(missing_summary)

# Identify which countries/years have missing values in core indices
missing_rows = df_vdem_raw[df_vdem_raw[['v2x_polyarchy', 'v2x_libdem']].isnull().any(axis=1)]
print(f"\nTotal rows with missing core indices: {len(missing_rows)}")
print("Countries affected:", missing_rows['country_name'].unique())

# make a copy of v-dem raw for processing
df_vdem = df_vdem_raw

# Isolate Bahrain's time series to inspect missing values
bahrain_mask = df_vdem['country_text_id'] == 'BHR'

# Create a display copy before imputation
bhr_before = df_vdem.loc[bahrain_mask, ['year', 'v2x_libdem']].copy()
bhr_before = bhr_before.rename(columns={'v2x_libdem': 'v2x_libdem_original'})

# Apply group-wise forward fill & backward fill
df_vdem['v2x_libdem'] = (
    df_vdem.groupby('country_text_id')['v2x_libdem']
    .transform(lambda g: g.ffill().bfill())
)

# Combine before & after columns for side-by-side verification
bhr_after = df_vdem.loc[bahrain_mask, ['year', 'v2x_libdem']].rename(columns={'v2x_libdem': 'v2x_libdem_imputed'})
comparison = pd.merge(bhr_before, bhr_after, on='year')

# Filter for the rows where imputation occurred plus surrounding years
missing_years = comparison[comparison['v2x_libdem_original'].isnull()]['year'].tolist()

if missing_years:
    min_year = max(min(missing_years) - 2, comparison['year'].min())
    max_year = min(max(missing_years) + 3, comparison['year'].max())

    print("--- Bahrain Imputation Verification ---")
    display(comparison[(comparison['year'] >= min_year) & (comparison['year'] <= max_year)])
else:
    print("No missing values found in original dataset for Bahrain!")


# ==========================================
# CLEAN FOREIGN ASSISTANCE DATASET
# ==========================================
# Total missing values per column
missing_summary = pd.DataFrame({
    'Missing Count': df_aid_raw.isnull().sum(),
    'Missing %': (df_aid_raw.isnull().mean() * 100).round(2)
})
print("--- Missing Values Summary ---")
print(missing_summary)

# Identify the exact column name for country code
# (Adjust 'Country Code' if the raw column is named 'country_code', 'Country ISO', etc.)
country_col = 'Country Code'

# Filter df_aid_raw for rows where the country code is missing
missing_country_rows = df_aid_raw[df_aid_raw['Country Code'].isnull()]

print(f"--- Rows with Missing {'Country Code'} ({len(missing_country_rows)} total) ---")
display(missing_country_rows)

# NOTE: Dropped 3 legacy foreign assistance obligation rows (2001–2002) labeled 'Czechoslovakia' due to post-1992 state dissolution and negligible dollar amounts. This exclusion ensures ISO code alignment with V-Dem country identifiers.

# Drop all rows where 'Country Code' is null/NaN
df_aid_clean = df_aid_raw.dropna(subset=['Country Code']).copy()

# Sanity check: verify 0 missing country codes remain
print(f"Remaining missing country codes: {df_aid_clean['Country Code'].isnull().sum()}")
print(f"Rows remaining in aid dataset: {len(df_aid_clean):,}")


# ==========================================
# FILTER & SELECT FOREIGN ASSISTANCE DATA
# ==========================================

# 1. Apply filtering conditions: DRG category and Obligations transaction type
drg_filter = df_aid_clean['US Category Name'] == 'Democracy, Human Rights, and Governance'
obligations_filter = df_aid_clean['Transaction Type Name'] == 'Obligations'

df_aid_filtered = df_aid_clean[drg_filter & obligations_filter].copy()

# 2. Select key columns for country-year panel analysis (constant_amount is lowercase in the dataset)
columns_to_keep = ['US Category Name', 'US Sector Name', 'Country Code', 'Country Name', 'Fiscal Year', 'constant_amount']
df_aid_subset = df_aid_filtered[columns_to_keep].copy()

# 3. Clean column names to lower_snake_case for seamless merging with V-Dem
df_aid_subset = df_aid_subset.rename(columns={
    'Country Code': 'country_text_id',
    'Fiscal Year': 'year',
    'constant_amount': 'aid_constant_amount'
})

# ==========================================
# VERIFICATION & SUMMARY
# ==========================================
print("--- Filtered Foreign Aid Dataset Summary ---")
print(f"Total DRG Obligation Records: {len(df_aid_subset):,}")
print(f"Year Range:                   {df_aid_subset['year'].min()} – {df_aid_subset['year'].max()}")
print(f"Unique Recipient Countries:   {df_aid_subset['country_text_id'].nunique()}")
print(f"Total Obligations (Constant): ${df_aid_subset['aid_constant_amount'].sum():,.2f}\n")

display(df_aid_subset.head())


# ==========================================
# 1. PRIMARY AGGREGATION (Country-Year Level)
# ==========================================
# Sum all DRG subsectors into a single total obligation figure per country-year
df_aid_aggregated = (
    df_aid_subset
    .groupby(['country_text_id', 'year'], as_index=False)['aid_constant_amount']
    .sum()
)

print("--- PRIMARY COUNTRY-YEAR PANEL ---")
print(f"Shape: {df_aid_aggregated.shape[0]:,} rows × {df_aid_aggregated.shape[1]} columns")
print(f"Unique Country-Year Observations: {len(df_aid_aggregated):,}")
print(f"Duplicate (country_text_id, year) Pairs: {df_aid_aggregated.duplicated(subset=['country_text_id', 'year']).sum()}")

# ==========================================
# 2. SUBSECTOR SUMMARY (For Optional EDA / Visualizations If Time Allows)
# ==========================================
# Group by US Sector / Subsector to see funding breakdown across the full time period

subsector_totals = (
    df_aid_subset
    .groupby('US Sector Name')['aid_constant_amount']
    .sum()
    .reset_index()
    .sort_values(by='aid_constant_amount', ascending=False)
)
subsector_totals['Share (%)'] = (subsector_totals['aid_constant_amount'] / subsector_totals['aid_constant_amount'].sum() * 100).round(2)

print("\n--- DRG SUBSECTOR BREAKDOWN (FOR EDA) ---")
display(subsector_totals)

display(df_aid_aggregated.head())

--- V-Dem (Democracy) Dataset Inspection ---
Row Count:     4,455
Column Count:  6
Memory Usage:  0.60 MB

Column Data Types & Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4455 entries, 0 to 4454
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country_name     4455 non-null   object 
 1   country_text_id  4455 non-null   object 
 2   year             4455 non-null   int64  
 3   v2x_polyarchy    4455 non-null   float64
 4   v2x_libdem       4452 non-null   float64
 5   v2x_cspart       4455 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 209.0+ KB
None

Head Preview:


,country_name,country_text_id,year,v2x_polyarchy,v2x_libdem,v2x_cspart
0,Mexico,MEX,1999,0.605,0.381,0.589
1,Mexico,MEX,2000,0.671,0.481,0.708
2,Mexico,MEX,2001,0.681,0.484,0.669



--- ForeignAssistance.gov (Aid Obligations) Dataset Inspection ---
Row Count:     189,242
Column Count:  11
Memory Usage:  66.15 MB

Column Data Types & Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189242 entries, 0 to 189241
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   US Category ID         189242 non-null  int64 
 1   US Category Name       189242 non-null  object
 2   US Sector ID           189242 non-null  int64 
 3   US Sector Name         189242 non-null  object
 4   Country Code           189235 non-null  object
 5   Country Name           189242 non-null  object
 6   Transaction Type ID    189242 non-null  int64 
 7   Transaction Type Name  189242 non-null  object
 8   Fiscal Year            189242 non-null  int64 
 9   current_amount         189242 non-null  int64 
 10  constant_amount        189242 non-null  int64 
dtypes: int64(6), object(5)
memory usage: 

,US Category ID,US Category Name,US Sector ID,US Sector Name,Country Code,Country Name,Transaction Type ID,Transaction Type Name,Fiscal Year,current_amount,constant_amount
0,1,Peace and Security,1,Counter-Terrorism,AFG,Afghanistan,1,Appropriated and Planned,2006,13320000,18623155
1,1,Peace and Security,1,Counter-Terrorism,AFG,Afghanistan,1,Appropriated and Planned,2007,21075000,28679216
2,1,Peace and Security,1,Counter-Terrorism,AFG,Afghanistan,1,Appropriated and Planned,2008,2876000,3833753



--- Missing Values Summary ---
                 Missing Count  Missing %
country_name                 0       0.00
country_text_id              0       0.00
year                         0       0.00
v2x_polyarchy                0       0.00
v2x_libdem                   3       0.07
v2x_cspart                   0       0.00

Total rows with missing core indices: 3
Countries affected: ['Bahrain']
--- Bahrain Imputation Verification ---


...

Remaining missing country codes: 0
Rows remaining in aid dataset: 189,235
--- Filtered Foreign Aid Dataset Summary ---
Total DRG Obligation Records: 10,040
Year Range:                   2001 – 2025
Unique Recipient Countries:   202
Total Obligations (Constant): $104,158,430,724.00



,US Category Name,US Sector Name,country_text_id,Country Name,year,aid_constant_amount
34771,"Democracy, Human Rights, and Governance",Rule of Law and Human Rights,AFG,Afghanistan,2002,104721704
34772,"Democracy, Human Rights, and Governance",Rule of Law and Human Rights,AFG,Afghanistan,2003,771636
34773,"Democracy, Human Rights, and Governance",Rule of Law and Human Rights,AFG,Afghanistan,2004,270334833
34774,"Democracy, Human Rights, and Governance",Rule of Law and Human Rights,AFG,Afghanistan,2005,5322387
34775,"Democracy, Human Rights, and Governance",Rule of Law and Human Rights,AFG,Afghanistan,2006,3291775


--- PRIMARY COUNTRY-YEAR PANEL ---
Shape: 3,736 rows × 3 columns
Unique Country-Year Observations: 3,736
Duplicate (country_text_id, year) Pairs: 0

--- DRG SUBSECTOR BREAKDOWN (FOR EDA) ---


,US Sector Name,aid_constant_amount,Share (%)
4,Rule of Law and Human Rights,51943760828,49.87
2,Good Governance,34166945724,32.80
0,Civil Society,12747861740,12.24
3,Political Competition and Consensus-Building,4922499319,4.73
1,"Democracy, Human Rights, and Governance - General",377363113,0.36


,country_text_id,year,aid_constant_amount
0,AFG,2001,134538
1,AFG,2002,105847567
2,AFG,2003,48604770
3,AFG,2004,517702753
4,AFG,2005,123433995


In [31]:
# ==========================================
# 1. GENERATE LAGS ON RAW V-DEM (1999-2023) FIRST
# ==========================================
df_vdem = df_vdem.sort_values(by=['country_text_id', 'year']).reset_index(drop=True)

index_cols = ['v2x_polyarchy', 'v2x_libdem', 'v2x_cspart']

for col in index_cols:
    df_vdem[f'{col}_lag1'] = df_vdem.groupby('country_text_id')[col].shift(1)
    df_vdem[f'{col}_lag2'] = df_vdem.groupby('country_text_id')[col].shift(2)

# ==========================================
# 2. MERGE WITH AID DATA (2001-2024)
# ==========================================
# Merge V-Dem panel (with lags pre-calculated) with aggregated aid
df_merged = pd.merge(
    df_vdem,
    df_aid_aggregated,
    on=['country_text_id', 'year'],
    how='outer'
)

# ==========================================
# 3. FILTER ANALYSIS WINDOW (2001-2024)
# ==========================================
# Restrict analysis to 2001-2024 (drops 1999 and 2000 rows completely)
df_panel = df_merged[(df_merged['year'] >= 2001) & (df_merged['year'] <= 2024)].copy()

# ==========================================
# 4. ZERO-FILL AID ONLY WITHIN 2001-2024
# ==========================================
# Within our active window, missing aid = $0 obligation
df_panel['aid_constant_amount'] = df_panel['aid_constant_amount'].fillna(0)

df_panel.head()

,country_name,country_text_id,year,v2x_polyarchy,v2x_libdem,v2x_cspart,v2x_polyarchy_lag1,v2x_polyarchy_lag2,v2x_libdem_lag1,v2x_libdem_lag2,v2x_cspart_lag1,v2x_cspart_lag2,aid_constant_amount
2,Afghanistan,AFG,2001,0.083,0.028,0.127,0.073,0.073,0.022,0.022,0.045,0.045,134538.0
3,Afghanistan,AFG,2002,0.218,0.085,0.615,0.083,0.073,0.028,0.022,0.127,0.045,105847567.0
4,Afghanistan,AFG,2003,0.224,0.085,0.637,0.218,0.083,0.085,0.028,0.615,0.127,48604770.0
5,Afghanistan,AFG,2004,0.235,0.089,0.638,0.224,0.218,0.085,0.085,0.637,0.615,517702753.0
6,Afghanistan,AFG,2005,0.311,0.111,0.665,0.235,0.224,0.089,0.085,0.638,0.637,123433995.0


In [32]:
# Ensure panel is sorted chronologically by country
df_panel = df_panel.sort_values(by=['country_text_id', 'year']).reset_index(drop=True)

# Group by country to calculate 1-year aid changes (delta)
df_panel['aid_delta'] = df_panel.groupby('country_text_id')['aid_constant_amount'].diff()

# Calculate 1-year changes for primary V-Dem democracy indices
index_cols = ['v2x_polyarchy', 'v2x_libdem', 'v2x_cspart']

for col in index_cols:
    # Delta of t-1 lagged score: Change in democracy between t-1 and t-2
    df_panel[f'{col}_lag1_delta'] = df_panel.groupby('country_text_id')[f'{col}_lag1'].diff()

    # Delta of contemporary score: Change in democracy between t and t-1
    df_panel[f'{col}_delta'] = df_panel.groupby('country_text_id')[col].diff()

# Quick verification display
delta_cols = ['country_text_id', 'year', 'aid_constant_amount', 'aid_delta', 'v2x_libdem_lag1', 'v2x_libdem_lag1_delta']
display(df_panel[delta_cols].head(10))

,country_text_id,year,aid_constant_amount,aid_delta,v2x_libdem_lag1,v2x_libdem_lag1_delta
0,AFG,2001,1.345380e+05,NaN,0.022,NaN
1,AFG,2002,1.058476e+08,105713029.0,0.028,0.006
2,AFG,2003,4.860477e+07,-57242797.0,0.085,0.057
3,AFG,2004,5.177028e+08,469097983.0,0.085,0.000
4,AFG,2005,1.234340e+08,-394268758.0,0.089,0.004
5,AFG,2006,8.953176e+07,-33902234.0,0.111,0.022
6,AFG,2007,2.267764e+08,137244601.0,0.192,0.081
7,AFG,2008,7.939467e+08,567170351.0,0.193,0.001
8,AFG,2009,1.597284e+09,803337366.0,0.196,0.003
9,AFG,2010,2.195716e+09,598431667.0,0.196,0.000


In [33]:
import os

# ==========================================
# 1. FINAL VARIABLE CHECK
# ==========================================
required_cols = [
    'country_name', 'country_text_id', 'year', 'aid_constant_amount', 'aid_delta',
    'v2x_polyarchy', 'v2x_polyarchy_lag1', 'v2x_polyarchy_lag2',
    'v2x_libdem', 'v2x_libdem_lag1', 'v2x_libdem_lag2',
    'v2x_cspart', 'v2x_cspart_lag1', 'v2x_cspart_lag2'
]

# Ensure panel has all required columns and is sorted
df_panel = df_panel[required_cols].sort_values(by=['country_text_id', 'year']).reset_index(drop=True)

# ==========================================
# 2. EXPORT PROCESSED PANEL CSV
# ==========================================
output_path = "../data/processed/merged_vdem_aid_panel.csv"

# Ensure the output directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_panel.to_csv(output_path, index=False)

print("--- EXPORT SUCCESSFUL ---")
print(f"File saved to: {output_path}")
print(f"Final shape:   {df_panel.shape[0]:,} rows × {df_panel.shape[1]} columns")
print(f"Years covered: {df_panel['year'].min()} – {df_panel['year'].max()}")
print(f"Countries:     {df_panel['country_text_id'].nunique()}")

display(df_panel.head())

--- EXPORT SUCCESSFUL ---
File saved to: ../data/processed/merged_vdem_aid_panel.csv
Final shape:   4,881 rows × 14 columns
Years covered: 2001 – 2024
Countries:     223


,country_name,country_text_id,year,aid_constant_amount,aid_delta,v2x_polyarchy,v2x_polyarchy_lag1,v2x_polyarchy_lag2,v2x_libdem,v2x_libdem_lag1,v2x_libdem_lag2,v2x_cspart,v2x_cspart_lag1,v2x_cspart_lag2
0,Afghanistan,AFG,2001,134538.0,NaN,0.083,0.073,0.073,0.028,0.022,0.022,0.127,0.045,0.045
1,Afghanistan,AFG,2002,105847567.0,105713029.0,0.218,0.083,0.073,0.085,0.028,0.022,0.615,0.127,0.045
2,Afghanistan,AFG,2003,48604770.0,-57242797.0,0.224,0.218,0.083,0.085,0.085,0.028,0.637,0.615,0.127
3,Afghanistan,AFG,2004,517702753.0,469097983.0,0.235,0.224,0.218,0.089,0.085,0.085,0.638,0.637,0.615
4,Afghanistan,AFG,2005,123433995.0,-394268758.0,0.311,0.235,0.224,0.111,0.089,0.085,0.665,0.638,0.637
